In [ ]:
import os
import json
import math
import random
import time
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


if not os.path.exists("FinQA"):
    !git clone https://github.com/czyssrs/FinQA.git


with open("FinQA/dataset/train.json", "r") as f:
    train_data = json.load(f)

with open("FinQA/dataset/dev.json", "r") as f:
    val_data = json.load(f)

with open("FinQA/dataset/test.json", "r") as f:
    test_data = json.load(f)


print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))


model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False


def format_finqa(example):

    table_text = "\n".join(
        [
            " | ".join(str(cell) for cell in row)
            for row in example["table"]
        ]
    )

    pre_text = "\n".join(
        str(x) for x in example["pre_text"]
    )

    post_text = "\n".join(
        str(x) for x in example["post_text"]
    )

    question = example["qa"]["question"]
    answer = example["qa"]["answer"]

    text = (
        "### Context:\n"
        f"{pre_text}\n\n"
        "### Table:\n"
        f"{table_text}\n\n"
        f"{post_text}\n\n"
        "### Question:\n"
        f"{question}\n\n"
        "### Answer:\n"
        f"{answer}"
    )

    return text


train_texts = [
    format_finqa(x)
    for x in train_data
]

val_texts = [
    format_finqa(x)
    for x in val_data
]

test_texts = [
    format_finqa(x)
    for x in test_data
]


train_dataset = Dataset.from_dict({
    "text": train_texts
})

val_dataset = Dataset.from_dict({
    "text": val_texts
})

test_dataset = Dataset.from_dict({
    "text": test_texts
})


MAX_LENGTH = 512


def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
)


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()


class ResearchLoggerCallback(TrainerCallback):

    def __init__(self):
        self.records = []

    def on_log(
        self,
        args,
        state,
        control,
        model=None,
        logs=None,
        **kwargs
    ):

        if logs is None:
            return

        record = {
            "step": state.global_step,
            "epoch": state.epoch,
            "loss": logs.get("loss"),
            "eval_loss": logs.get("eval_loss"),
            "learning_rate": logs.get("learning_rate"),
            "grad_norm": logs.get("grad_norm"),
        }

        adapter_sq_sum = 0.0

        if model is not None:

            for name, param in model.named_parameters():

                if (
                    "lora_" in name
                    and param.requires_grad
                ):

                    param_norm = (
                        param.detach()
                        .float()
                        .norm(2)
                        .item()
                    )

                    adapter_sq_sum += (
                        param_norm ** 2
                    )

        record["adapter_norm"] = math.sqrt(
            adapter_sq_sum
        )

        if torch.cuda.is_available():

            record["gpu_memory_allocated_mb"] = (
                torch.cuda.memory_allocated()
                / 1024**2
            )

            record["gpu_memory_reserved_mb"] = (
                torch.cuda.memory_reserved()
                / 1024**2
            )

            record["gpu_max_memory_allocated_mb"] = (
                torch.cuda.max_memory_allocated()
                / 1024**2
            )

        else:

            record["gpu_memory_allocated_mb"] = None
            record["gpu_memory_reserved_mb"] = None
            record["gpu_max_memory_allocated_mb"] = None

        self.records.append(record)


research_logger = ResearchLoggerCallback()


training_args = TrainingArguments(

    output_dir="./lora_finqa",

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    learning_rate=1e-4,

    fp16=True,

    logging_strategy="steps",

    logging_steps=50,

    eval_strategy="steps",

    eval_steps=250,

    save_strategy="steps",

    save_steps=250,

    save_total_limit=10,

    seed=SEED,

    gradient_checkpointing=True,

    report_to="none",

    remove_unused_columns=False,
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[research_logger],
)


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time


print(
    f"Training time: {training_time / 3600:.2f} hours"
)


with open(
    "./lora_finqa/training_time.txt",
    "w"
) as f:

    f.write(
        f"{training_time:.2f} seconds\n"
    )

    f.write(
        f"{training_time / 3600:.2f} hours\n"
    )


print("LoRA training completed.")


final_model_path = "./lora_finqa/final_model"

trainer.save_model(
    final_model_path
)

tokenizer.save_pretrained(
    final_model_path
)


signals_df = pd.DataFrame(
    research_logger.records
)

signals_path = (
    "./lora_finqa/training_signals.csv"
)

signals_df.to_csv(
    signals_path,
    index=False
)


print("\nTraining signals:")
print(signals_df.head())

print(
    "\nSignals saved to:",
    signals_path
)


results = trainer.evaluate()

print(
    "LoRA Validation Loss:",
    results["eval_loss"]
)